In [ ]:
import requests
import pandas as pd
import os

LABELS = ["WALKING", "STANDING", "SITTING", "LAYING", "WALKING_UPSTAIRS", "WALKING_DOWNSTAIRS"]

def call_prediction_api(window_vals, label_counts):
    url = "http://127.0.0.1:8006/predict"
    data = {"samples": window_vals.to_dict(orient="records")}
    print(data)
    response = requests.post(url, json=data)
    if response.status_code == 200:
        print("API Response:", response.json())
        predicted = response.json()["activity"]

        row = {label: label_counts.get(label, 0) for label in LABELS}
        row["predicted"] = predicted

        csv_path = "prediction_report_validation_set.csv"
        file_exists = os.path.exists(csv_path)
        pd.DataFrame([row]).to_csv(csv_path, mode="a", header=not file_exists, index=False)

    else:
        print("API Error:", response.status_code, response.text)


In [22]:

import pandas as pd
import time

data_path = r"/home/pitoleo/src/neat-calculator/neat_dashboard/validation_set.csv"

df = pd.read_csv(data_path)

# Keep the label column to save with windows
# df = df.drop(columns=['label'])


window = 128
step = 64
index = 0

X_windows = []


for start in range(0, len(df) - window + 1, step):
    end = start + window
    window_vals = df[start:end].copy()
    
    # Replace timestamps with current time + 20ms increments
    current_time_ms = int(time.time() * 1000)
    for i in range(len(window_vals)):
        window_vals.iloc[i, window_vals.columns.get_loc('timestamp')] = current_time_ms + (i * 20)
    
    # Add window_id to track which window this belongs to
    # window_vals['window_id'] = index
    
    X_windows.append(window_vals)

    # print(X_windows[0])
    label_counts = window_vals["label"].value_counts()
    print(window_vals["label"].value_counts().to_string())
    call_prediction_api(window_vals, label_counts)

    time.sleep(0.02)
    
    index += 1
    

# print(f"Created {len(X_windows)} windows of size {window} with step {step} out of {len(df)} data points.")

# # Concatenate all windows into a single DataFrame
# all_windows_df = pd.concat(X_windows, ignore_index=True)

# # Group by window_id and get the most prevalent label for each window
# if 'label' in all_windows_df.columns:
#     window_labels = all_windows_df.groupby('window_id')['label'].agg(lambda x: x.mode()[0] if len(x.mode()) > 0 else x.iloc[0])
#     window_summary = pd.DataFrame({
#         'window_id': window_labels.index,
#         'label': window_labels.values
#     })
# else:
#     window_summary = pd.DataFrame({
#         'window_id': all_windows_df['window_id'].unique()
#     })

# window_summary.to_csv(r"E:\src\neat-calculator\neat_dashboard\windows.csv", index=False)

# print(f"Saved {len(window_summary)} windows with their labels to windows.csv")


label
WALKING_DOWNSTAIRS    128
{'samples': [{'accelerometerX': -2.58, 'accelerometerY': -8.67, 'accelerometerZ': -1.74, 'gyroscopeX': -0.56, 'gyroscopeY': -0.25, 'gyroscopeZ': -0.26999998, 'timestamp': 1772563491775, 'timestampNanos': 10387230887053, 'label': 'WALKING_DOWNSTAIRS'}, {'accelerometerX': -3.62, 'accelerometerY': -5.2599998, 'accelerometerZ': -3.97, 'gyroscopeX': -0.53999996, 'gyroscopeY': -3.0, 'gyroscopeZ': -1.04, 'timestamp': 1772563491795, 'timestampNanos': 10387277642210, 'label': 'WALKING_DOWNSTAIRS'}, {'accelerometerX': -5.18, 'accelerometerY': -4.47, 'accelerometerZ': -0.62, 'gyroscopeX': -0.14, 'gyroscopeY': -2.6799998, 'gyroscopeZ': -0.58, 'timestamp': 1772563491815, 'timestampNanos': 10387298850751, 'label': 'WALKING_DOWNSTAIRS'}, {'accelerometerX': -4.99, 'accelerometerY': -5.3599997, 'accelerometerZ': -1.74, 'gyroscopeX': -0.07, 'gyroscopeY': 0.55, 'gyroscopeZ': 0.45, 'timestamp': 1772563491835, 'timestampNanos': 10387318024345, 'label': 'WALKING_DOWNSTAIRS'},